# Synthetic data and masking exports

Purpose: deterministic synthetic images and bounded spatial multiblock masks for Notebook 00.
Inputs: grid, areas, aspect ratios, batch size, and explicit CPU `torch.Generator`.
Outputs: images and original-index masks. Runtime: definitions only, under one second; sampling is small-grid CPU work.
Checkpoint location: none in this definitions notebook; the orchestrator stores generator state in `checkpoints/<run_id>/`.
Resume: run this notebook top to bottom to define the API, or import only its `ahr-export` cell through the package bootstrap. Restore the generator state from the orchestrator checkpoint before drawing the next batch.


In [ ]:
from pathlib import Path
import json, os
PROJECT_ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).expanduser().resolve()
_bootstrap = json.loads((PROJECT_ROOT / "fmm_jepa/bootstrap.ipynb").read_text())
_bootstrap_namespace = {}
for _cell in _bootstrap["cells"]:
    if _cell["cell_type"] == "code" and "ahr-export" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), "bootstrap.ipynb#ahr-export", "exec"), _bootstrap_namespace)
_bootstrap_namespace["install_notebook_package"](PROJECT_ROOT)


In [ ]:
"""Synthetic inputs and spatial masks; all definitions are imported by tagged exports."""
import math
import torch


def _range(name, bounds, *, positive=True, ceiling=None):
    if len(bounds) != 2 or bounds[0] > bounds[1] or (positive and bounds[0] <= 0):
        raise ValueError(f"invalid {name}: {bounds}")
    if ceiling is not None and bounds[1] > ceiling:
        raise ValueError(f"{name} must not exceed {ceiling}")


def _uniform(bounds, generator):
    return bounds[0] + (bounds[1] - bounds[0]) * torch.rand((), generator=generator).item()


def _rectangle(grid, area, aspect, generator):
    rows, cols = grid
    count = rows * cols * _uniform(area, generator)
    ratio = math.exp(_uniform((math.log(aspect[0]), math.log(aspect[1])), generator))
    height = max(1, min(rows, round(math.sqrt(count / ratio))))
    width = max(1, min(cols, round(math.sqrt(count * ratio))))
    top = int(torch.randint(rows - height + 1, (), generator=generator))
    left = int(torch.randint(cols - width + 1, (), generator=generator))
    return [r * cols + c for r in range(top, top + height) for c in range(left, left + width)]


def _pack(blocks):
    width = max(map(len, blocks))
    indices = torch.full((len(blocks), width), -1, dtype=torch.long)
    valid = torch.zeros_like(indices, dtype=torch.bool)
    for b, block in enumerate(blocks):
        indices[b, :len(block)] = torch.tensor(block, dtype=torch.long)
        valid[b, :len(block)] = True
    return indices, valid


def sample_masks(grid, batch_size, generator, context_area=(0.65, 0.90),
                 target_area=(0.08, 0.18), aspect_ratio=(0.5, 2.0), target_count=2,
                 max_retries=32):
    """Draw CPU multiblock masks. Indices are original row-major patch indices.

    Each target block is independently sampled, so overlap across target blocks
    is intentionally allowed. One random anchor is excluded from all targets,
    guaranteeing an available context token. Context is a sampled rectangle
    minus the union of targets. After bounded retries, targets fall back to a
    singleton other than the anchor and context falls back to the anchor.
    Rounded/clipped rectangles can differ from the requested area/aspect range;
    these ranges govern proposals, not an exact post-discretization guarantee.
    """
    rows, cols = map(int, grid)
    if rows < 1 or cols < 1 or rows * cols < 2 or batch_size < 1 or target_count < 1 or max_retries < 1:
        raise ValueError("need a grid of >=2 patches, positive batch/target counts and retries")
    if str(generator.device) != "cpu":
        raise ValueError("mask generator must be CPU; move returned tensors afterward")
    _range("context_area", context_area, ceiling=1)
    _range("target_area", target_area, ceiling=1)
    _range("aspect_ratio", aspect_ratio)
    contexts, targets = [], [[] for _ in range(target_count)]
    fallback_targets = fallback_contexts = 0
    for _ in range(batch_size):
        anchor = int(torch.randint(rows * cols, (), generator=generator))
        hidden = set()
        for k in range(target_count):
            for attempt in range(max_retries):
                block = _rectangle((rows, cols), target_area, aspect_ratio, generator)
                if anchor not in block:
                    break
            else:
                block = [(anchor + 1) % (rows * cols)]
                fallback_targets += 1
            targets[k].append(block)
            hidden.update(block)
        for attempt in range(max_retries):
            proposal = _rectangle((rows, cols), context_area, aspect_ratio, generator)
            visible = [i for i in proposal if i not in hidden]
            if visible:
                break
        else:
            visible = [anchor]
            fallback_contexts += 1
        contexts.append(visible)
    context_indices, context_valid = _pack(contexts)
    packed = [_pack(blocks) for blocks in targets]
    return {"context_indices": context_indices, "context_valid": context_valid,
            "target_indices": [item[0] for item in packed],
            "target_valid": [item[1] for item in packed],
            "metadata": {"grid": [rows, cols], "target_overlap_allowed": True,
                         "fallback_target_blocks": fallback_targets,
                         "fallback_contexts": fallback_contexts, "max_retries": max_retries}}


def synthetic_images(batch_size, height, width, generator):
    """CPU float32 BCHW synthetic Gaussian blobs, stripes and noise in [0,1]."""
    if batch_size < 1 or height < 1 or width < 1:
        raise ValueError("positive image dimensions required")
    if str(generator.device) != "cpu":
        raise ValueError("synthetic input generator must be CPU")
    yy, xx = torch.meshgrid(torch.linspace(0, 1, height), torch.linspace(0, 1, width), indexing="ij")
    centers = torch.rand((batch_size, 3, 2, 1, 1), generator=generator)
    scales = 0.025 + 0.12 * torch.rand((batch_size, 3, 1, 1), generator=generator)
    blobs = torch.exp(-((xx - centers[:, :, 0]) ** 2 + (yy - centers[:, :, 1]) ** 2) / scales)
    frequency = 1 + 4 * torch.rand((batch_size, 3, 1, 1), generator=generator)
    phase = 2 * math.pi * torch.rand((batch_size, 3, 1, 1), generator=generator)
    stripes = 0.5 + 0.5 * torch.sin(2 * math.pi * frequency * (xx + 0.6 * yy) + phase)
    noise = 0.05 * torch.rand((batch_size, 3, height, width), generator=generator)
    return (0.55 * blobs + 0.40 * stripes + noise).clamp(0, 1)
